# Alcampo Web Scraper

This notebook collects product and price information from the Alcampo online store using **Selenium and BeautifulSoup**.

1. Runs the selected searches and loads additional results by scrolling.
2. Extracts product name, ID, price, unit price, URL and collection timestamp.
3. Combines and deduplicates the records, then checks collection quality.
4. Saves dated **RAW**, **PROCESSED** and **QUALITY** snapshots in `data/history/alcampo/`.
5. Updates the current CSV files only after a complete, valid full run. Partial runs can be archived with their quality status but do not replace the current data.

Use `TEST_MODE = True` for one search (`leche`), without saving files. Switch to `False` for all 20 searches. A `stable` scroll status means no more items appeared after several scrolls; it does **not** prove the full catalogue was retrieved. Existing historical files are never overwritten.


In [17]:
import time
import re
from pathlib import Path
from datetime import datetime
from urllib.parse import urljoin, quote

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException


HOME_URL = "https://www.compraonline.alcampo.es/"

SEARCH_TERMS = [
    "leche",
    "huevos",
    "manzana",
    "plátano",
    "tomate",
    "arroz",
    "pasta",
    "pan",
    "pollo",
    "carne picada",
    "salmón",
    "yogur",
    "queso",
    "agua",
    "zumo",
    "atún",
    "garbanzos",
    "aceite",
    "detergente",
    "papel higiénico",
]

# --------------------------------------------------
# TEST MODE
# --------------------------------------------------

TEST_MODE = False
TEST_TERMS = ["leche"]

TERMS_TO_SCRAPE = TEST_TERMS if TEST_MODE else SEARCH_TERMS

MAX_SCROLLS = 40
SCROLL_PAUSE = 1.5
STABLE_ROUNDS_TO_STOP = 3

print("Search terms:", TERMS_TO_SCRAPE)

Search terms: ['leche', 'huevos', 'manzana', 'plátano', 'tomate', 'arroz', 'pasta', 'pan', 'pollo', 'carne picada', 'salmón', 'yogur', 'queso', 'agua', 'zumo', 'atún', 'garbanzos', 'aceite', 'detergente', 'papel higiénico']


In [ ]:
# --------------------------------------------------
# PROJECT DIRECTORIES
# --------------------------------------------------

def find_project_root():
    current = Path.cwd().resolve()
    for path in (current, *current.parents):
        if path.name == "supermarket_project":
            return path
    if current.name == "scrapers":
        return current.parent
    raise FileNotFoundError("Project root not found. Run from the scrapers directory.")

BASE_DIR = find_project_root()
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
HISTORY_DIR = BASE_DIR / "data" / "history" / "alcampo"

for directory in (RAW_DIR, PROCESSED_DIR, HISTORY_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Keep these filenames unchanged for compatibility with the existing pipeline.
RAW_PATH = RAW_DIR / "alcampo_products_raw.csv"
PROCESSED_PATH = PROCESSED_DIR / "alcampo_products.csv"
RUN_DATE = datetime.now().strftime("%Y-%m-%d")
RAW_HISTORY_PATH = HISTORY_DIR / f"{RUN_DATE}_raw.csv"
PROCESSED_HISTORY_PATH = HISTORY_DIR / f"{RUN_DATE}_processed.csv"

print("Project root:", BASE_DIR)
print("Collection date:", RUN_DATE)
print("History directory:", HISTORY_DIR)


Project root: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project
Collection date: 2026-11-10
History directory: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\history\alcampo


In [19]:
# --------------------------------------------------
# BROWSER AND NAVIGATION HELPERS
# --------------------------------------------------

from selenium.webdriver.chrome.options import Options
from selenium import webdriver

def start_alcampo_driver():
    options = Options()
    options.add_argument("--window-size=1920,1080")
    
    # Browser configuration
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")
    
    driver = webdriver.Chrome(options=options)
    
    # Browser compatibility setting
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    
    driver.get(HOME_URL)
    return driver


def accept_alcampo_cookies(driver):
    xpaths = [
        "//button[contains(., 'Aceptar todas')]",
        "//button[contains(., 'Aceptar todo')]",
        "//button[contains(., 'Aceptar')]",
        "//button[contains(., 'Permitir todas')]",
    ]

    for xpath in xpaths:
        buttons = driver.find_elements(By.XPATH, xpath)

        for button in buttons:
            if button.is_displayed() and button.is_enabled():
                try:
                    button.click()
                    print("Cookies accepted.")
                    return
                except Exception:
                    pass

    print("No visible cookie banner.")


def assert_no_security_challenge(driver):
    body_text = driver.find_element(By.TAG_NAME, "body").text.lower()

    messages = [
        "captcha",
        "access denied",
        "confirma que no eres un robot",
        "comprueba que eres humano",
        "verify you are human",
    ]

    if any(message in body_text for message in messages):
        raise RuntimeError(
            "A security challenge was detected."
        )


def find_search_box(driver):
    # Try semantic attributes first.
    selectors = [
        'input[type="search"]',
        'input[placeholder*="Buscar en Alcampo"]',
        'input[placeholder*="Buscar"]',
        'input[aria-label*="Buscar"]',
        'input[name*="search"]',
        'input[id*="search"]',
    ]

    def visible_search_box():
        for selector in selectors:
            elements = driver.find_elements(By.CSS_SELECTOR, selector)

            for element in elements:
                if element.is_displayed() and element.is_enabled():
                    return element

        return False

    try:
        return WebDriverWait(driver, 12).until(
            lambda _: visible_search_box()
        )

    except TimeoutException:
        diagnostics = []

        for element in driver.find_elements(By.TAG_NAME, "input"):
            if element.is_displayed():
                diagnostics.append({
                    "type": element.get_attribute("type"),
                    "name": element.get_attribute("name"),
                    "id": element.get_attribute("id"),
                    "placeholder": element.get_attribute("placeholder"),
                    "aria-label": element.get_attribute("aria-label"),
                })

        raise RuntimeError(
            "Could not find the Alcampo search input. "
            f"Visible inputs: {diagnostics}"
        )

In [20]:
# --------------------------------------------------
# PRODUCT SEARCH
# --------------------------------------------------

def count_product_links(driver):
    links = driver.find_elements(
        By.CSS_SELECTOR,
        'a[href*="/products/"]'
    )

    hrefs = {
        link.get_attribute("href")
        for link in links
        if link.get_attribute("href")
    }

    return len(hrefs)


def search_alcampo(driver, term):
    assert_no_security_challenge(driver)
    
    encoded_term = quote(term)
    # Use the website search URL
    search_url = f"https://www.compraonline.alcampo.es/search?q={encoded_term}"
    
    print(f"Opening search URL: {search_url}")
    driver.get(search_url)
    
    time.sleep(4)  # Allow time for the page to render
    assert_no_security_challenge(driver)
    
    # Report the current page for debugging
    print("Current page title:", driver.title)
    print("Product links detected:", count_product_links(driver))

    def search_loaded(d):
        assert_no_security_challenge(d)
        
        # Wait until product links appear
        new_count = count_product_links(d)
        if new_count > 0:
            return True
            
        return False

    try:
        WebDriverWait(
            driver,
            20,
            poll_frequency=0.4
        ).until(search_loaded)

    except TimeoutException:
        # A search can legitimately return no results.
        body_text = driver.find_element(By.TAG_NAME, "body").text.lower()

        if (
            "no hemos encontrado" in body_text
            or "sin resultados" in body_text
            or "0 resultados" in body_text
        ):
            print(f"No results for: {term}")
            return

        raise TimeoutException(
            f"Could not confirm the search page loaded for '{term}'."
        )

    print(
        f"Search completed: {term}\n"
        f"Current URL: {driver.current_url}"
    )

In [21]:
# --------------------------------------------------
# LOAD SEARCH RESULTS BY SCROLLING
# --------------------------------------------------


# --------------------------------------------------
# LOAD RESULTS PROGRESSIVELY
# --------------------------------------------------

def load_all_alcampo_products(driver, searched_item, max_steps=60):

    collected = {}
    no_new_at_bottom = 0

    def capture_visible_products():
        df = extract_alcampo_products(
            driver, searched_item, verbose=False
        )

        if df.empty:
            return 0

        before = len(collected)

        for record in df.to_dict("records"):
            url = record.get("url")
            if url:
                collected.setdefault(url, record)

        return len(collected) - before

    # Capture products initially visible
    capture_visible_products()

    print(f"Initial products collected: {len(collected)}")

    for step in range(1, max_steps + 1):

        assert_no_security_challenge(driver)

        # Scroll gradually instead of jumping to the bottom
        driver.execute_script(
            "window.scrollBy(0, Math.max(180, "
            "Math.floor(window.innerHeight * 0.6)));"
        )

        time.sleep(1.2)

        # Capture products after each scroll
        new_count = capture_visible_products()

        position, viewport, page_height = driver.execute_script(
            "return [window.scrollY, window.innerHeight, "
            "document.documentElement.scrollHeight];"
        )

        at_bottom = position + viewport >= page_height - 100

        if step == 1 or new_count or step % 10 == 0:
            print(
                f"Step {step}: +{new_count} products | "
                f"Total: {len(collected)}"
            )

        if at_bottom and new_count == 0:
            no_new_at_bottom += 1
        else:
            no_new_at_bottom = 0

        if no_new_at_bottom >= 6:
            print("End of results reached.")
            return pd.DataFrame(
                list(collected.values())
            ), "stable"

    print("Maximum scroll steps reached.")

    return pd.DataFrame(
        list(collected.values())
    ), "max_scrolls_reached"


In [22]:
# --------------------------------------------------
# DATA CLEANING HELPERS
# --------------------------------------------------

def clean_number(text):
    if text is None:
        return None

    match = re.search(
        r"\d+(?:[.,]\d+)?",
        str(text).replace("\xa0", " ")
    )

    if not match:
        return None

    return float(
        match.group()
        .replace(",", ".")
    )


def normalize_unit_name(unit_text):
    if not unit_text:
        return None

    unit = (
        unit_text
        .strip()
        .lower()
        .replace(".", "")
    )

    mappings = {
        "kg": "KILO",
        "kilo": "KILO",
        "kilogramo": "KILO",
        "kilogramos": "KILO",

        "l": "LITRO",
        "litro": "LITRO",
        "litros": "LITRO",

        "unidad": "UNIDAD",
        "ud": "UNIDAD",
        "uds": "UNIDAD",

        "docena": "DOCENA",
        "lavado": "LAVADO",
        "lavados": "LAVADO",
    }

    return mappings.get(unit, unit.upper())


def extract_price_per_unit(text):
    if not text:
        return None, None, None

    cleaned = (
        str(text)
        .replace("\xa0", " ")
        .strip()
    )

    # Typical format:
    # 1000ml(0,83 € por litro)
    match = re.search(
        r"(\d+(?:[.,]\d+)?)\s*€\s+por\s+"
        r"(litro|litros|kg|kilo|kilogramo|kilogramos|"
        r"unidad|ud|uds|docena|lavado|lavados)",
        cleaned,
        flags=re.IGNORECASE,
    )

    if match:
        value = float(
            match.group(1)
            .replace(",", ".")
        )

        unit = normalize_unit_name(
            match.group(2)
        )

        return value, unit, match.group(0)

    # Posibles formatos por 100 ml / 100 g
    match_100 = re.search(
        r"(\d+(?:[.,]\d+)?)\s*€\s*(?:por|/)\s*100\s*"
        r"(ml|g|gr)",
        cleaned,
        flags=re.IGNORECASE,
    )

    if match_100:
        value = float(
            match_100.group(1)
            .replace(",", ".")
        )

        measure = match_100.group(2).lower()

        if measure == "ml":
            return value * 10, "LITRO", match_100.group(0)

        return value * 10, "KILO", match_100.group(0)

    return None, None, None


def extract_product_id(url):
    if not url:
        return None

    match = re.search(
        r"/products/[^/]+/(\d+)",
        url
    )

    return match.group(1) if match else None

In [23]:
# --------------------------------------------------
# EXTRACT PRODUCTS FROM THE CURRENT PAGE
# --------------------------------------------------

def find_product_container(anchor):
    current = anchor

    for _ in range(8):
        current = current.parent

        if current is None:
            break

        text = current.get_text(" ", strip=True)

        # Product cards usually contain a price
        # and an add-to-cart button.
        if (
            "Precio" in text
            and "Añadir" in text
            and len(text) < 2500
        ):
            return current

    return anchor.parent


def extract_alcampo_products(driver, searched_item, verbose=True):
    soup = BeautifulSoup(
        driver.page_source,
        "html.parser"
    )

    anchors = soup.select(
        'a[href*="/products/"]'
    )

    records = []
    seen_urls = set()
    scraped_at = datetime.now().isoformat(
        timespec="seconds"
    )

    for anchor in anchors:
        href = anchor.get("href")

        if not href:
            continue

        url = urljoin(HOME_URL, href)

        # The same product may have more than one link
        # inside its card.
        if url in seen_urls:
            continue

        seen_urls.add(url)

        container = find_product_container(anchor)

        if container is None:
            continue

        text = container.get_text(
            " ",
            strip=True
        )

        # --------------------------------------------------
        # PRODUCT NAME
        # --------------------------------------------------

        name = (
            anchor.get("aria-label")
            or anchor.get("title")
            or anchor.get_text(" ", strip=True)
        )

        # If the anchor has no usable text,
        # look for a title inside the product card.
        if not name:
            candidate = container.find(
                ["h2", "h3", "h4"]
            )

            if candidate:
                name = candidate.get_text(
                    " ",
                    strip=True
                )

        # --------------------------------------------------
        # PRODUCT PRICE
        # --------------------------------------------------

        price = None
        price_raw = None

        price_match = re.search(
            r"Precio\s+(\d+(?:[.,]\d+)?)\s*€",
            text,
            flags=re.IGNORECASE,
        )

        if price_match:
            price_raw = price_match.group(0)
            price = float(
                price_match.group(1)
                .replace(",", ".")
            )

        # --------------------------------------------------
        # PRODUCT PRICE POR UNIDAD
        # --------------------------------------------------

        price_per_unit, unit, ppu_raw = (
            extract_price_per_unit(text)
        )

        # --------------------------------------------------
        # PACKAGE SIZE
        # --------------------------------------------------

        quantity_raw = None

        quantity_match = re.search(
            r"\b\d+(?:[.,]\d+)?\s*"
            r"(?:ml|cl|l|g|gr|kg|uds?|unidades?)\b",
            text,
            flags=re.IGNORECASE,
        )

        if quantity_match:
            quantity_raw = quantity_match.group(0)

        product_id = extract_product_id(url)

        # Skip entries without required product information.
        
        if not name or price is None or price <= 0:
            continue

        records.append({
            "searched_item": searched_item,
            "supermarket": "Alcampo",
            "product_id": product_id,
            "name": name,
            "quantity_raw": quantity_raw,
            "price_raw": price_raw,
            "price": price,
            "price_per_unit_raw": ppu_raw,
            "price_per_unit": price_per_unit,
            "unit": unit,
            "url": url,
            "scraped_at": scraped_at,
        })

    df = pd.DataFrame(records)

    if verbose:
        print(f"Product links detected: {len(anchors)}")
        print(f"Products extracted: {len(df)}")

    return df

In [24]:
# --------------------------------------------------
# COLLECT PRODUCT DATA
# --------------------------------------------------

all_results = []
summary = []

driver = start_alcampo_driver()

try:
    time.sleep(3)
    accept_alcampo_cookies(driver)

    for i, term in enumerate(
        TERMS_TO_SCRAPE,
        start=1
    ):
        print(
            "\n" + "=" * 70
        )
        print(
            f"[{i}/{len(TERMS_TO_SCRAPE)}] "
            f"Searching: {term}"
        )

        start = time.monotonic()

        try:
            search_alcampo(
                driver,
                term
            )

            df_term, load_status = load_all_alcampo_products(driver, term)

            loaded_count = len(df_term)

            elapsed = round(
                time.monotonic() - start,
                2
            )

            all_results.append(
                df_term
            )

            summary.append({
                "searched_item": term,
                "status": load_status,
                "products_loaded": loaded_count,
                "rows_extracted": len(df_term),
                "seconds": elapsed,
            })

            print(
                f"✓ {len(df_term)} rows extracted "
                f"in {elapsed} s"
            )

            time.sleep(2)

        except Exception as exc:
            elapsed = round(
                time.monotonic() - start,
                2
            )

            print(
                f"ERROR for '{term}': "
                f"{type(exc).__name__}: {exc}"
            )

            summary.append({
                "searched_item": term,
                "status": "error",
                "products_loaded": None,
                "rows_extracted": 0,
                "seconds": elapsed,
            })

            # Stop after a search error; flag the run as partial.
            break

finally:
    driver.quit()


df_alcampo = (
    pd.concat(
        all_results,
        ignore_index=True
    )
    if all_results
    else pd.DataFrame()
)

df_summary = pd.DataFrame(
    summary
)

display(df_summary)

print(
    "\nTotal raw rows:",
    len(df_alcampo)
)

Cookies accepted.

[1/20] Searching: leche
Opening search URL: https://www.compraonline.alcampo.es/search?q=leche
Current page title: Resultados de la búsqueda para leche - Alcampo supermercado online
Product links detected: 5
Search completed: leche
Current URL: https://www.compraonline.alcampo.es/search?q=leche
Initial products collected: 5
Step 1: +5 products | Total: 10
Step 2: +5 products | Total: 15
Step 3: +5 products | Total: 20
Step 4: +5 products | Total: 25
Step 6: +5 products | Total: 30
Step 7: +5 products | Total: 35
Step 8: +5 products | Total: 40
Step 9: +1 products | Total: 41
Step 10: +0 products | Total: 41
Step 11: +5 products | Total: 46
Step 12: +5 products | Total: 51
Step 13: +5 products | Total: 56
Step 14: +5 products | Total: 61
Step 16: +5 products | Total: 66
Step 17: +5 products | Total: 71
Step 18: +5 products | Total: 76
Step 19: +5 products | Total: 81
Step 20: +0 products | Total: 81
Step 21: +5 products | Total: 86
Step 22: +5 products | Total: 91
Ste

,searched_item,status,products_loaded,rows_extracted,seconds
0,leche,max_scrolls_reached,236,236,227.64
1,huevos,stable,87,87,82.61
2,manzana,stable,200,200,177.31
3,plátano,stable,136,136,120.42
4,tomate,stable,172,172,138.03
5,arroz,stable,203,203,158.59
6,pasta,stable,79,79,85.56
7,pan,stable,41,41,66.20
8,pollo,stable,48,48,78.83
9,carne picada,stable,78,78,73.75



Total raw rows: 1869


In [25]:
# --------------------------------------------------
# RAW DATA REVIEW
# --------------------------------------------------

if not df_alcampo.empty:
    display(
        df_alcampo.head(30)
    )

    print(
        "Rows without product_id:",
        df_alcampo["product_id"]
        .isna()
        .sum()
    )

    print(
        "Distinct product IDs:",
        df_alcampo["product_id"]
        .nunique()
    )

    print("\nRows per search:")

    display(
        df_alcampo
        .groupby("searched_item")
        .size()
        .rename("rows")
    )

else:
    print(
        "No data to review."
    )

,searched_item,supermarket,product_id,name,quantity_raw,price_raw,price,price_per_unit_raw,price_per_unit,unit,url,scraped_at
0,leche,Alcampo,54178,AUCHAN Leche entera de vaca 6 x 1 l Producto A...,1 l,"Precio 5,76 €",5.76,"0,96 € por litro",0.96,LITRO,https://www.compraonline.alcampo.es/products/a...,2026-11-10T13:40:43
1,leche,Alcampo,54180,AUCHAN Leche semidesnatada de vaca 6 x 1l Prod...,1l,"Precio 5,28 €",5.28,"0,88 € por litro",0.88,LITRO,https://www.compraonline.alcampo.es/products/a...,2026-11-10T13:40:43
2,leche,Alcampo,99193,L.R. Leche entera 6 x 1 l.,1 l,"Precio 5,70 €",5.70,"0,95 € por litro",0.95,LITRO,https://www.compraonline.alcampo.es/products/l...,2026-11-10T13:40:43
3,leche,Alcampo,53082,CENTRAL LECHERA ASTURIANA Leche de vaca semide...,1 l,"Precio 6,24 €",6.24,"1,04 € por litro",1.04,LITRO,https://www.compraonline.alcampo.es/products/c...,2026-11-10T13:40:43
4,leche,Alcampo,216880,AUCHAN Leche mini entera 6 x 200 ml Producto A...,200 ml,"Precio 1,71 €",1.71,"1,42 € por litro",1.42,LITRO,https://www.compraonline.alcampo.es/products/a...,2026-11-10T13:40:43
5,leche,Alcampo,53004,CENTRAL LECHERA ASTURIANA Leche de vaca entera...,1l,"Precio 7,08 €",7.08,"1,18 € por litro",1.18,LITRO,https://www.compraonline.alcampo.es/products/c...,2026-11-10T13:40:45
6,leche,Alcampo,53540,AUCHAN Leche entera de vaca 1 l. Producto Alcampo,1 l,"Precio 0,96 €",0.96,"0,96 € por litro",0.96,LITRO,https://www.compraonline.alcampo.es/products/a...,2026-11-10T13:40:45
7,leche,Alcampo,54434,PRÉSIDENT Leche entera de vaca de origen 100% ...,1 l,"Precio 6,43 €",6.43,"1,07 € por litro",1.07,LITRO,https://www.compraonline.alcampo.es/products/p...,2026-11-10T13:40:45
8,leche,Alcampo,54497,PASCUAL Leche entera procedente de animales co...,1l,"Precio 7,38 €",7.38,"1,23 € por litro",1.23,LITRO,https://www.compraonline.alcampo.es/products/p...,2026-11-10T13:40:45
9,leche,Alcampo,53096,PULEVA EcoLeche entera pack de 6 uds x 1 l.,6 uds,"Precio 8,26 €",8.26,"1,38 € por litro",1.38,LITRO,https://www.compraonline.alcampo.es/products/p...,2026-11-10T13:40:45


Rows without product_id: 0
Distinct product IDs: 1780

Rows per search:


searched_item
aceite              49
agua                49
arroz              203
atún                50
carne picada        78
detergente          48
garbanzos           51
huevos              87
leche              236
manzana            200
pan                 41
papel higiénico     58
pasta               79
plátano            136
pollo               48
queso               53
salmón              77
tomate             172
yogur              106
zumo                48
Name: rows, dtype: int64

In [26]:
# --------------------------------------------------
# PROCESSED DATA: DEDUPLICATION
# --------------------------------------------------

if not df_alcampo.empty:

    df_alcampo_clean = (
        df_alcampo.copy()
    )

    print(
        "Raw rows:",
        len(df_alcampo_clean)
    )

    # Prefer product IDs for deduplication.
    # If a product ID is missing,
    # use name, price and package size to deduplicate.

    with_id = (
        df_alcampo_clean[
            df_alcampo_clean[
                "product_id"
            ].notna()
        ]
        .drop_duplicates(
            subset=["product_id"]
        )
    )

    without_id = (
        df_alcampo_clean[
            df_alcampo_clean[
                "product_id"
            ].isna()
        ]
        .drop_duplicates(
            subset=[
                "name",
                "price",
                "quantity_raw"
            ]
        )
    )

    df_alcampo_clean = (
        pd.concat(
            [with_id, without_id],
            ignore_index=True
        )
        .reset_index(drop=True)
    )

    print(
        "Unique products:",
        len(df_alcampo_clean)
    )

    print(
        "Unique product IDs:",
        df_alcampo_clean[
            "product_id"
        ].nunique()
    )

else:
    df_alcampo_clean = pd.DataFrame()

Raw rows: 1869
Unique products: 1780
Unique product IDs: 1780


In [27]:
# --------------------------------------------------
# DUPLICATE REVIEW
# --------------------------------------------------

if not df_alcampo_clean.empty:

    duplicated_ids = (
        df_alcampo_clean[
            df_alcampo_clean[
                "product_id"
            ].notna()
            & df_alcampo_clean.duplicated(
                subset=["product_id"],
                keep=False
            )
        ]
    )

    possible_duplicates = (
        df_alcampo_clean[
            df_alcampo_clean.duplicated(
                subset=["name", "price"],
                keep=False
            )
        ]
        .sort_values(
            ["name", "price"]
        )
    )

    print(
        "Rows with duplicate product IDs:",
        len(duplicated_ids)
    )

    print(
        "Rows sharing name and price:",
        len(possible_duplicates)
    )

    if not possible_duplicates.empty:
        display(
            possible_duplicates[
                [
                    "product_id",
                    "name",
                    "price",
                    "price_per_unit",
                    "unit",
                    "url",
                ]
            ].head(100)
        )

Rows with duplicate product IDs: 0
Rows sharing name and price: 0


In [28]:
# --------------------------------------------------
# UNIT REVIEW
# --------------------------------------------------

if not df_alcampo_clean.empty:

    print(
        df_alcampo_clean["unit"]
        .value_counts(
            dropna=False
        )
    )

    unknown_units = (
        df_alcampo_clean[
            df_alcampo_clean[
                "price_per_unit"
            ].isna()
        ]
    )

    print(
        "\nProducts without detected unit price:",
        len(unknown_units)
    )

    display(
        unknown_units[
            [
                "name",
                "quantity_raw",
                "price",
                "price_per_unit_raw",
                "unit",
                "url",
            ]
        ].head(50)
    )

unit
KILO      1173
LITRO      426
NaN         88
None        58
DOCENA      35
Name: count, dtype: int64

Products without detected unit price: 146


,name,quantity_raw,price,price_per_unit_raw,unit,url
315,Recipiente con capacidad para 12 huevos A FUEG...,NaN,2.99,NaN,NaN,https://www.compraonline.alcampo.es/products/r...
316,"Set de cuece huevos doble para microondas, var...",NaN,2.99,NaN,NaN,https://www.compraonline.alcampo.es/products/s...
317,"Soporte para huevos en espiral, huevera de ace...",NaN,1.25,NaN,NaN,https://www.compraonline.alcampo.es/products/s...
320,"Huevera 6 huevos, juguete Happy para niños. ON...",NaN,5.99,NaN,NaN,https://www.compraonline.alcampo.es/products/h...
352,PRODUCTO ALCAMPO Chips de manzana 30 g.,30 g,1.10,NaN,NaN,https://www.compraonline.alcampo.es/products/p...
354,SMILEAT Triboo Snack (con formas divertidas) d...,25 g,1.64,NaN,NaN,https://www.compraonline.alcampo.es/products/s...
358,SMILEAT Triboo Snacks (con formas divertidas) ...,25 g,1.64,NaN,NaN,https://www.compraonline.alcampo.es/products/s...
382,SMILEAT Smilitos Snacks a base de cereales eco...,25 g,1.50,NaN,NaN,https://www.compraonline.alcampo.es/products/s...
405,SMILEAT Tribbo Snack (de formas divertidas) de...,25 g,1.64,NaN,NaN,https://www.compraonline.alcampo.es/products/s...
410,SMILEAT Tribbo Snack (de formas divertidas) de...,25 g,1.64,NaN,NaN,https://www.compraonline.alcampo.es/products/s...


In [ ]:

# --------------------------------------------------
# VALIDATE AND SAVE CURRENT AND HISTORICAL DATA
# --------------------------------------------------

import os
import tempfile

if TEST_MODE:
    print("TEST MODE: no files will be saved.")

else:
    # Validate collected datasets
    if df_alcampo.empty or df_alcampo_clean.empty:
        raise RuntimeError(
            "No usable products collected. No files saved."
        )

    if df_summary.empty:
        raise RuntimeError(
            "No search results were recorded."
        )

    # Check whether all search terms were completed
    all_terms_attempted = (
        len(df_summary) == len(SEARCH_TERMS)
        and set(df_summary["searched_item"]) == set(SEARCH_TERMS)
    )

    successful_searches = df_summary["status"].eq("stable").all()

    collection_complete = bool(
        all_terms_attempted and successful_searches
    )

    # Validate prices
    if (
        df_alcampo_clean["price"].isna().any()
        or (df_alcampo_clean["price"] <= 0).any()
    ):
        raise RuntimeError(
            "Missing or invalid prices detected."
        )

    # Validate product names
    if df_alcampo_clean["name"].isna().any():
        raise RuntimeError(
            "Missing product names detected."
        )

    # --------------------------------------------------
    # CHECK HISTORICAL FILES
    # --------------------------------------------------

    history_outputs = [
        (RAW_HISTORY_PATH, df_alcampo),
        (PROCESSED_HISTORY_PATH, df_alcampo_clean),
    ]

    existing = [
        str(path)
        for path, _ in history_outputs
        if path.exists()
    ]

    if existing:
        raise FileExistsError(
            "Historical files already exist for this date: "
            + ", ".join(existing)
        )

    # --------------------------------------------------
    # PREPARE CSV FILES
    # --------------------------------------------------

    def prepare_csv_temp(frame, destination):
        fd, tmp_name = tempfile.mkstemp(
            prefix=".alcampo_",
            suffix=".tmp",
            dir=destination.parent
        )

        try:
            with os.fdopen(
                fd, "w", encoding="utf-8-sig", newline=""
            ) as f:
                frame.to_csv(f, index=False)

            return Path(tmp_name)

        except Exception:
            Path(tmp_name).unlink(missing_ok=True)
            raise

    # Always save historical data
    outputs = list(history_outputs)

    # Update current files only for complete collections
    if collection_complete:
        outputs.extend([
            (RAW_PATH, df_alcampo),
            (PROCESSED_PATH, df_alcampo_clean),
        ])

    # --------------------------------------------------
    # SAVE FILES
    # --------------------------------------------------

    staged = []

    try:
        for destination, frame in outputs:
            staged.append((
                prepare_csv_temp(frame, destination),
                destination
            ))

        for tmp, destination in staged:
            os.replace(tmp, destination)
            print("Saved:", destination)

    finally:
        for tmp, _ in staged:
            tmp.unlink(missing_ok=True)

    # --------------------------------------------------
    # COLLECTION SUMMARY
    # --------------------------------------------------

    if collection_complete:
        print(
            "Full collection validated. "
            "Current CSV files updated."
        )
    else:
        print(
            "WARNING: Partial collection saved in history. "
            "Current CSV files unchanged."
        )

        display(
            df_summary.loc[
                df_summary["status"] != "stable",
                ["searched_item", "status", "rows_extracted"]
            ]
        )

    print("Collection date:", RUN_DATE)
    print("Raw rows:", len(df_alcampo))
    print("Processed products:", len(df_alcampo_clean))


Saved: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\history\alcampo\2026-11-10_raw.csv
Saved: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\history\alcampo\2026-11-10_processed.csv
Saved: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\history\alcampo\2026-11-10_quality.csv


,searched_item,status,rows_extracted
0,leche,max_scrolls_reached,236


Raw rows: 1869
Processed products: 1780
